In [2]:
# grab directory root
import sys
sys.path.append("../")

In [3]:
from dinov3.eval.tSNE import extract_embeddings
from dinov3.data.datasets import NCells
from dinov3.models.backbone_loader import load_backbone
from dinov3.eval.simpleKNN import evaluate_simple_knn
import torch
import json
from torchvision import transforms


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/site-packages/torch/distributed/distributed_c10d.py:4631: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
  warnings.warn(  # warn only once
[rank0]:[W916 18:11:45.527527252 ProcessGroupNCCL.cpp:4718] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 as device used by this process is currently unknown. This can potentially cause a hang if this rank to GPU mapping is incorrect. You can pecify device_id in init_process_group() to force use of a particular device.


In [4]:
# defaul transform used in dinov3
def make_transform(resize_size: int | list[int] = 768):
    to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([to_tensor, resize, normalize])

test_dataset = NCells(
    root="/home/students/code/helmholtzSS25/Hannes/dinov3playground/manifest_test_fixed.csv.gz",
    split=NCells.Split.TRAIN,
    transform=make_transform(), 
    target_transform=None,
)
print(f"Test Dataset contains {len(test_dataset)} entries")

Test Dataset contains 359426 entries


# BASELINE

In [7]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80/ckpt/123749",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 123750 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        

batches: 100%|██████████| 5617/5617 [06:55<00:00, 13.53it/s]


In [8]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='euclidean', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5214300376838424,
        "precision@k": 0.47329352912699696,
        "recall@k": 0.00021737782546046302,
        "topk_acc": 0.47329352912699696
    },
    "5": {
        "mAP": 0.5214300376838424,
        "precision@k": 0.4577732273124371,
        "recall@k": 0.0009739680009373416,
        "topk_acc": 0.6500141892907024
    },
    "10": {
        "mAP": 0.5214300376838424,
        "precision@k": 0.4492340565234568,
        "recall@k": 0.0018230845973724002,
        "topk_acc": 0.7289762009426141
    }
}


In [6]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5931701375128093,
        "precision@k": 0.47364408807376207,
        "recall@k": 0.00022098644261653267,
        "topk_acc": 0.47364408807376207
    },
    "5": {
        "mAP": 0.5931701375128093,
        "precision@k": 0.5153077406754102,
        "recall@k": 0.0009799491545323365,
        "topk_acc": 0.7874833762721672
    },
    "10": {
        "mAP": 0.5931701375128093,
        "precision@k": 0.49975627806558226,
        "recall@k": 0.0018330538328638808,
        "topk_acc": 0.8676166999604925
    }
}


# Hierarchical dbscan

In [5]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [09:54<00:00,  9.44it/s]


In [1]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

NameError: name 'evaluate_simple_knn' is not defined

In [ ]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='euclidean', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

In [8]:
from dinov3.eval.OG_simpleKNN import evaluate_simple_knn


In [9]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("og kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

og kNN Performance: {
    "1": {
        "mAP": 0.6014779925281668,
        "precision@k": 0.5451608954277097,
        "recall@k": 0.000220863487129198,
        "top1_acc": 0.5451608954277097
    },
    "5": {
        "mAP": 0.6014779925281668,
        "precision@k": 0.5296111021461997,
        "recall@k": 0.0009798198642552653,
        "top1_acc": 0.5451608954277097
    },
    "10": {
        "mAP": 0.6014779925281668,
        "precision@k": 0.5069085152437498,
        "recall@k": 0.0018329176020889077,
        "top1_acc": 0.5451608954277097
    }
}


In [5]:
from dinov3.eval.simpleKNN import evaluate_simple_knn